In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib

In [2]:
# Configurar cadena de conexión
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=comerssiamirror.eastus2.cloudapp.azure.com,38693;"
    "DATABASE=PROVENZAL;"
    "UID=provenzal;"
    "PWD=PE@4:BRy/<1VWkp;"
)

# Crear el motor
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [11]:
query = """
SELECT SUBSTRING(Cliente,2, LEN(Cliente))AS Cliente,
    Fecha,
    SUM(Venta_Neta) AS Venta,
    Canal, Ubicacion
FROM Ventas_MoviNova
WHERE Fecha > '2025-06-01'
    AND CANAL != 'C&C'
GROUP BY Cliente, Fecha, Canal, Ubicacion
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query, engine)
df_ventas.head()


,Cliente,Fecha,Venta,Canal,Ubicacion
0,000040401865,2025-06-05,94957.98,Retail,LOCCITANE TESORO
1,01048391,2025-06-13,274789.91,Retail,LOCCITANE GRAN ESTACION
2,0292691,2025-06-22,155462.18,Retail,LOCCITANE PARQUE LA COLINA
3,0400596680,2025-06-14,275630.26,Retail,LOCCITANE TESORO
4,1000149406,2025-06-25,66386.55,Retail,LOCCITANE ANDINO


In [14]:
df_aperturas = pd.read_excel("Atribucion.xlsx")
df_aperturas['Cliente'] = df_aperturas['Cliente'].astype(str)

df_merge = df_aperturas.merge(df_ventas, on='Cliente', how='inner')
df_merge['dias_despues'] = (df_merge['Fecha'] - df_merge['sentAt']).dt.days

df_atribuidas = df_merge[(df_merge['dias_despues'] >= 0) & (df_merge['dias_despues'] <= 5)]

venta_total_atribuida = df_atribuidas['Venta'].sum()

print(f"💰 Venta total atribuida a campañas: ${venta_total_atribuida:,.2f}")

💰 Venta total atribuida a campañas: $315,126.05


In [17]:
ventas_por_cliente = df_atribuidas.groupby(['Cliente', 'sentAt'])['Venta'].sum().reset_index()

df_aperturas_con_venta = df_aperturas.merge(
    ventas_por_cliente, 
    on=['Cliente', 'sentAt'], 
    how='left'
)

# 3. Reemplazar NaN por 0 para los que no compraron
df_aperturas_con_venta['Venta'] = df_aperturas_con_venta['Venta'].fillna(0)

# Exportar todos los registros con validaciones incluidas
df_aperturas_con_venta.to_excel("clientes_con_validacion3.xlsx", index=False)

In [7]:
df.to_sql("Contacto_Clientes", engine, if_exists="replace", index=False)

12